In [11]:
def overlap(a, b, min_length):
    for length in range(len(a), min_length - 1, -1):
        if a[-length:] == b[:length]:
            return length
    return 0

In [23]:
def overlap_imp(a, b, min_length):
    if a.find(b[:min_length]) == -1:
        return 0
    start = 0
    while True:
        if start > len(a) - min_length:
            return 0
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start

        start += 1

In [24]:
print(overlap_imp("ACGAAACGT", "ACGTGGG", 3))
print(overlap("ACGAAACGT", "ACGTGGG", 3))
print(overlap_imp("ATGCA", "ATGGG", 3))

4
4
0


---

## Verification (test harness offered by Claude)

The code cells from here onward were offered by Claude (Anthropic), not
written by me. They are not part of the solution — they only verify the
work above: a correctness sweep checking `overlap_imp` against the
brute-force `overlap`, and a timing comparison of the two.

The `overlap` and `overlap_imp` algorithms above are my own original work.

In [25]:
import random

# --- Test harness (offered by Claude) ---


def random_read(length):
    """Return a random DNA string of the given length."""
    return "".join(random.choices("ACGT", k=length))


def correctness_sweep(trials=5000):
    """Check overlap_imp against the brute-force overlap across many inputs."""

    # Known-answer regression cases: (a, b, min_length, expected)
    regression = [
        ("ACGAAACGT", "ACGTGGG", 3, 4),
        ("AAAAT", "AAATGGG", 3, 4),
        ("ATGCA", "ATGGG", 3, 0),
        ("ACGTAA", "GTACGT", 3, 0),
    ]
    for a, b, ml, expected in regression:
        got = overlap_imp(a, b, ml)
        if got != expected:
            print(
                f"REGRESSION FAIL: overlap_imp({a!r}, {b!r}, {ml}) "
                f"= {got}, expected {expected}"
            )
            return

    # Random sweep: overlap_imp must always agree with the brute-force oracle.
    for _ in range(trials):
        a = random_read(random.randint(8, 15))
        b = random_read(random.randint(8, 15))
        ml = random.randint(2, 5)
        slow = overlap(a, b, ml)
        fast = overlap_imp(a, b, ml)
        if slow != fast:
            print(
                f"MISMATCH: a={a!r} b={b!r} ml={ml} "
                f"-> overlap={slow}, overlap_imp={fast}"
            )
            return

    print(f"Passed: {len(regression)} regression cases + {trials} random pairs.")


correctness_sweep()

Passed: 4 regression cases + 5000 random pairs.


In [26]:
# --- Timing comparison (offered by Claude) ---


def make_read_batch(n_reads=200, read_length=100):
    """Return a list of random DNA reads."""
    return [random_read(read_length) for _ in range(n_reads)]


def all_pairs_run(overlap_fn, reads, min_length):
    """Run overlap_fn over every ordered pair of distinct reads; count the hits."""
    count = 0
    for a in reads:
        for b in reads:
            if a is b:  # skip a read against itself
                continue
            if overlap_fn(a, b, min_length) > 0:
                count += 1
    return count


reads = make_read_batch(n_reads=200, read_length=100)
MIN_LENGTH = 30

In [27]:
%timeit all_pairs_run(overlap, reads, MIN_LENGTH)

140 ms ± 857 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [28]:
%timeit all_pairs_run(overlap_imp, reads, MIN_LENGTH)

6.44 ms ± 21 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
